# Módulo 03 — Limpeza e Tratamento de Dados Reais com `Pandas`

## 1. Por que Limpar Dados é Vital na Engenharia e em IA?

Em projetos reais de engenharia e Aprendizado de Máquina, **até 80% do trabalho acontece antes de ligar a Rede Neural**: sensores perdem comunicação, operadores digitam `'?'` ou `'-'` onde faltou leitura, e números acabam salvos como texto.

Se alimentarmos uma Rede Neural com um único valor ausente (`NaN` — *Not a Number*, ou "Não é um Número"), todas as multiplicações de matrizes resultarão em `NaN`, inutilizando o treinamento!

Vamos carregar novamente `arquivos-csv/titanic.csv` e investigar por que `age` (idade) e `fare` (tarifa) vieram "disfarçadas" de texto:

In [ ]:
import numpy as np
import pandas as pd

# Carregando a tabela bruta do Titanic
df = pd.read_csv("arquivos-csv/titanic.csv")

# Verificando os tipos originais das colunas 'age', 'fare' e 'embarked'
print(df[["age", "fare", "embarked"]].dtypes)
print("\nExemplos de valores estranhos na coluna 'age':")
print(df[df["age"] == "?"][["id", "name", "age", "cabin"]].head(4))

**Diagnóstico:** Encontramos o culpado! Quando os dados foram catalogados, valores desconhecidos receberam o texto `'?'`. Para o `Pandas` inicialmente pareceram colunas 100% preenchidas, mas na verdade `'?'` é um **dado faltante mascarado**.

---

## 2. Padronizando Valores Ausentes (`replace` e `isna`)

O primeiro passo da faxina é trocar todo `'?'` por `np.nan` (o marcador numérico padrão de ausência) e contar exatamente **quantos valores faltam em cada coluna** usando `df.isna().sum()`.

In [ ]:
# Substitui todas as ocorrências da string '?' pelo marcador numérico padrão np.nan
df = df.replace("?", np.nan)

# Conta quantos valores ausentes (NaN) existem agora em cada coluna
faltantes_por_coluna = df.isna().sum()
print("Quantidade de valores ausentes (NaN) por coluna:")
print(faltantes_por_coluna[faltantes_por_coluna > 0])

**Análise Quantitativa:**
- `embarked` perdeu apenas `2` registros e `fare` perdeu apenas `1` registro (fáceis de recuperar!).
- `age` tem `263` valores ausentes (~20% da base — vale muito a pena recuperar em vez de jogar fora 263 passageiros!).
- Já `cabin` (`1014` ausentes) e `body` (`1188` ausentes) faltam em quase toda a base (e `boat`/`body` nem poderiam ser usados para prever sobrevivência, pois só existem *depois* do naufrágio — um caso clássico de **Vazamento de Dados / *Data Leakage***!).

---

## 3. Convertendo Textos para Números (`pd.to_numeric`)

Mesmo após trocar `'?'` por `np.nan`, as colunas `age` e `fare` continuam com o rótulo de texto (`object`). Se tentarmos calcular a média agora sem converter, o Python lançará um `TypeError` (não se soma texto!). Veja na prática usando `try...except` (estrutura que captura erros sem travar o caderno):

In [ ]:
# Demonstrando de forma segura o erro que ocorre antes de converter o tipo da coluna
try:
    media_teste = df["age"].mean()
    print("Média calculada:", media_teste)
except TypeError as erro:
    print("Erro capturado (esperado!):", erro)

Para corrigir isso definitivamente, aplicamos `pd.to_numeric(coluna, errors='coerce')`:
- O argumento `errors='coerce'` (*forçar conversão*) transforma `"29"` no número decimal `29.0` e converte automaticamente qualquer resíduo inválido em `NaN`.

In [ ]:
# Convertendo 'age' e 'fare' de texto para ponto flutuante (float64)
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")

print("Novos tipos de dados:")
print(df[["age", "fare"]].dtypes)
print(f"Média de idade (ignorando NaNs): {df['age'].mean():.2f} anos")
print(f"Mediana da tarifa (ignorando NaNs): £ {df['fare'].median():.2f}")

---

## 4. Estratégias de Imputação (`fillna`): Média, Mediana ou Moda?

Como preencher (**imputar**) os buracos `NaN` para não perder linhas valiosas?
1. **Variáveis Numéricas Simétricas:** Preenchemos com a **média** (`.mean()`).
2. **Variáveis Numéricas com *Outliers* (valores extremos, como tarifas milionárias):** A **mediana** (`.median()`, o valor central que divide a amostra ao meio) é muito mais robusta que a média!
3. **Variáveis Categóricas (textos como o porto `embarked`):** Preenchemos com a **moda** (`.mode()[0]`, a categoria mais frequente).

> **Regra de Ouro de Machine Learning (Zero Vazamento de Dados):** Quando formos treinar uma Rede Neural dividindo dados em **Treino** e **Teste**, sempre calcularemos a média/mediana/moda **apenas no conjunto de Treino** e usaremos esse valor fixo para preencher o Teste!

In [ ]:
# 1. Preenchendo colunas numéricas ('age' com mediana ou média; aqui testamos mediana para 'fare' e média para 'age')
df["age"] = df["age"].fillna(df["age"].mean())      # Substitui NaN em 'age' pela média (29.88 anos)
df["fare"] = df["fare"].fillna(df["fare"].median())  # Substitui NaN em 'fare' pela mediana (£ 14.45)

# 2. Preenchendo coluna categórica 'embarked' com o porto mais frequente (moda)
porto_mais_comum = df["embarked"].mode()[0]          # [0] pega o 1º valor caso haja empate
df["embarked"] = df["embarked"].fillna(porto_mais_comum)

# Conferindo se ainda resta algum NaN nas três colunas tratadas
print("Valores ausentes restantes após limpeza:")
print(df[["age", "fare", "embarked"]].isna().sum())

**Resultado:** Zero valores ausentes em `age`, `fare` e `embarked`, com tipos numéricos prontos para alimentar gráficos estatísticos e Redes Neurais!

---

## 5. Exercícios Práticos de Fixação

### Exercício 1: Carregamento Direto Limpo com `na_values`
Sabendo de antemão que um arquivo CSV usa `'?'` para representar falhas de leitura, o próprio `pd.read_csv()` aceita o parâmetro `na_values=['?']`, que já converte `'?'` em `NaN` e detecta `age` e `fare` como `float64` automaticamente! Teste isso lendo `arquivos-csv/titanic.csv` em `df_limpo` e exiba `df_limpo[['age', 'fare']].dtypes`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

# Passando na_values=['?'], o Pandas faz a limpeza e a conversão numérica de uma só vez!
df_limpo = pd.read_csv("arquivos-csv/titanic.csv", na_values=["?"])

print("Tipos detectados automaticamente:")
print(df_limpo[["age", "fare"]].dtypes)
print("Faltantes em age e fare:", df_limpo[["age", "fare"]].isna().sum().to_dict())
```

</details>

### Exercício 2: Média vs. Mediana na Presença de *Outliers*
Em `arquivos-csv/titanic.csv` (lido com `na_values=['?']`), compare o valor da **média** (`mean()`) e da **mediana** (`median()`) da tarifa `fare`. Por que elas são tão diferentes? Imprima também a tarifa máxima (`max()`).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

df_limpo = pd.read_csv("arquivos-csv/titanic.csv", na_values=["?"])

media_tarifa = df_limpo["fare"].mean()
mediana_tarifa = df_limpo["fare"].median()
max_tarifa = df_limpo["fare"].max()

print(f"Média: £ {media_tarifa:.2f} | Mediana: £ {mediana_tarifa:.2f} | Máximo: £ {max_tarifa:.2f}")
# Explicação: Poucos bilhetes de suítes de luxo (£ 512.33) puxam a média para £ 33.30,
# enquanto 50% dos passageiros pagaram até £ 14.45 (mediana)!
```

</details>

### Exercício 3: Codificação Numérica de Variável Textual (`map`)
Redes Neurais só multiplicam números, nunca palavras como `'male'` ou `'female'`. Crie uma coluna `sexo_num` mapeando `'female' -> 0` e `'male' -> 1` com `df_limpo['sex'].map({'female': 0, 'male': 1})` e verifique as 5 primeiras linhas.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

df_limpo = pd.read_csv("arquivos-csv/titanic.csv", na_values=["?"])

# Traduz cada texto do dicionário para seu código binário correspondente (0 ou 1)
df_limpo["sexo_num"] = df_limpo["sex"].map({"female": 0, "male": 1})

print(df_limpo[["sex", "sexo_num"]].head(5))
```

</details>

---

## Próximo Passo

Com os dados devidamente limpos e tipados, chegou a hora de enxergar padrões, distribuições e *outliers* usando gráficos científicos no **[Módulo 04 — Visualização de Dados com Matplotlib e Seaborn](04_visualizacao_dados.ipynb)**!